# Parte 2: Transformación a CNF

En esta parte implementarás las transformaciones necesarias para convertir fórmulas a **Forma Normal Conjuntiva (CNF)**.

## ¿Qué es CNF?

Una fórmula está en CNF si es una conjunción (∧) de cláusulas, donde cada cláusula es una disyunción (∨) de literales.

**Ejemplo:** `(p ∨ ¬q) ∧ (¬p ∨ r ∨ s) ∧ (q)`

## Pipeline de conversión a CNF

1. Eliminar bicondicionales (↔) → **TÚ**
2. Eliminar implicaciones (→) → **TÚ**
3. Mover negaciones hacia adentro (De Morgan) → **TÚ**
4. Eliminar dobles negaciones (¬¬) → **DADO**
5. Distribuir ∨ sobre ∧ → **TÚ**
6. Aplanar → **TÚ**

In [1]:
import sys

sys.path.insert(0, "..")

from src.logic.propositional.ast import Atom, Not, And, Or, Implies, Iff
from src.logic.propositional.utils import formula_to_string

## Paso 1: Eliminar Bicondicionales

**Regla:** `A ↔ B` ≡ `(A → B) ∧ (B → A)`

Implementa `eliminate_iff` en `src/logic/propositional/cnf.py`.

In [2]:
from src.logic.propositional.cnf import eliminate_iff

f = Iff(Atom("p"), Atom("q"))
result = eliminate_iff(f)
print(f"Original: {formula_to_string(f)}")
print(f"Sin ↔:    {formula_to_string(result)}")

Original: (p ↔ q)
Sin ↔:    ((p → q) ∧ (q → p))


## Paso 2: Eliminar Implicaciones

**Regla:** `A → B` ≡ `¬A ∨ B`

Implementa `eliminate_implication` en `src/logic/propositional/cnf.py`.

In [3]:
from src.logic.propositional.cnf import eliminate_implication

f = Implies(Atom("p"), Atom("q"))
result = eliminate_implication(f)
print(f"Original: {formula_to_string(f)}")
print(f"Sin →:    {formula_to_string(result)}")

Original: (p → q)
Sin →:    (¬p ∨ q)


## Paso 3: Mover Negaciones hacia Adentro (De Morgan)

**Reglas:**
- `¬(A ∧ B)` ≡ `¬A ∨ ¬B`
- `¬(A ∨ B)` ≡ `¬A ∧ ¬B`

Implementa `push_negation_inward` en `src/logic/propositional/cnf.py`.

In [ ]:
from src.logic.propositional.cnf import push_negation_inward

# De Morgan sobre And
f1 = Not(And(Atom("p"), Atom("q")))
r1 = push_negation_inward(f1)
print(f"¬(p ∧ q):  {formula_to_string(f1)}")
print(f"De Morgan: {formula_to_string(r1)}")

print()

# De Morgan sobre Or
f2 = Not(Or(Atom("p"), Atom("q")))
r2 = push_negation_inward(f2)
print(f"¬(p ∨ q):  {formula_to_string(f2)}")
print(f"De Morgan: {formula_to_string(r2)}")

## Paso 4: Eliminar Dobles Negaciones (DADO)

**Regla:** `¬¬A` ≡ `A`

Esta función ya está implementada. Puedes estudiarla como referencia para las demás.

In [ ]:
from src.logic.propositional.cnf import eliminate_double_negation

f = Not(Not(Atom("p")))
result = eliminate_double_negation(f)
print(f"Original: {formula_to_string(f)}")
print(f"Sin ¬¬:   {formula_to_string(result)}")

## Paso 5: Distribuir ∨ sobre ∧

**Regla:** `A ∨ (B ∧ C)` ≡ `(A ∨ B) ∧ (A ∨ C)`

Esta es la transformación clave para obtener CNF: cada ∨ que contenga un ∧ adentro se "reparte".

Implementa `distribute_or_over_and` en `src/logic/propositional/cnf.py`.

In [ ]:
from src.logic.propositional.cnf import distribute_or_over_and

# p ∨ (q ∧ r) → (p ∨ q) ∧ (p ∨ r)
f = Or(Atom("p"), And(Atom("q"), Atom("r")))
result = distribute_or_over_and(f)
print(f"Original:    {formula_to_string(f)}")
print(f"Distribuido: {formula_to_string(result)}")

## Paso 6: Aplanar

**Reglas:**
- `(A ∧ B) ∧ C` ≡ `A ∧ B ∧ C`
- `(A ∨ B) ∨ C` ≡ `A ∨ B ∨ C`

El último paso: eliminar anidamientos innecesarios para obtener una CNF limpia.

Implementa `flatten` en `src/logic/propositional/cnf.py`.

In [ ]:
from src.logic.propositional.cnf import flatten

# And(And(a, b), c) → And(a, b, c)
f = And(And(Atom("a"), Atom("b")), Atom("c"))
result = flatten(f)
print(f"Original:  {formula_to_string(f)}")
print(f"Aplanado:  {formula_to_string(result)}")

## Pipeline completo: `to_cnf`

Una vez implementadas tus 5 funciones, el pipeline completo debería funcionar.

In [ ]:
from src.logic.propositional.cnf import to_cnf

# p → (q ∧ r) debería dar (¬p ∨ q) ∧ (¬p ∨ r)
f = Implies(Atom("p"), And(Atom("q"), Atom("r")))
cnf = to_cnf(f)
print(f"Original: {formula_to_string(f)}")
print(f"CNF:      {formula_to_string(cnf)}")

## CNF con un escenario FIFA

Una fórmula anidada con átomos del Mundial puede verse larga; al convertirla a CNF suele quedar una única cláusula. Aquí se puede observar el resultado de la transformación.

In [ ]:
# Implicaciones anidadas: se leen "de afuera hacia adentro",
# pero en CNF quedan como una sola disyunción de literales.
f = Implies(
    Atom("MEX_vence_RSA"),
    Implies(
        Atom("MEX_vence_KOR"),
        Implies(Atom("MEX_9pts"), Atom("MEX_clasifica")),
    ),
)
cnf = to_cnf(f)

print(f"Original: {formula_to_string(f)}")
print(f"CNF:      {formula_to_string(cnf)}")

## Verificar tu implementación

Ejecuta las pruebas del Punto 2:

```bash
uv run pytest tests/test_cnf.py -v
```

O la celda siguiente.

In [ ]:
# Verificar implementación CNF
!cd .. && uv run pytest tests/test_cnf.py -v